# 10. MoE, routing, and DeepSeek-V4 MTP

Only tensor widths, vocabulary size, batch size, and sequence length are reduced.
V4 keeps 256 routed experts/top-6/+1 shared and K3 keeps 896/top-16/+2.
The MTP depth reuses the same V4 decoder-block structure instead of substituting a simpler block.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(7)
device = torch.device('cpu')
torch.set_num_threads(min(2, torch.get_num_threads()))


In [ ]:
class BatchedSwiGLUExperts(nn.Module):

    def __init__(self, expert_count, input_dim, intermediate_dim, output_dim, clamp_limit=10.0):
        super().__init__()
        self.clamp_limit = clamp_limit
        scale = 0.02
        self.gate_weight = nn.Parameter(
            scale * torch.randn(expert_count, input_dim, intermediate_dim)
        )
        self.value_weight = nn.Parameter(
            scale * torch.randn(expert_count, input_dim, intermediate_dim)
        )
        self.output_weight = nn.Parameter(
            scale * torch.randn(expert_count, intermediate_dim, output_dim)
        )

    def selected_forward(self, hidden, expert_ids):
        gate_weight = self.gate_weight[expert_ids]
        value_weight = self.value_weight[expert_ids]
        output_weight = self.output_weight[expert_ids]
        gate = torch.einsum('bti,btkif->btkf', hidden, gate_weight)
        value = torch.einsum('bti,btkif->btkf', hidden, value_weight)
        gate = gate.clamp(max=self.clamp_limit)
        value = value.clamp(-self.clamp_limit, self.clamp_limit)
        hidden_expert = F.silu(gate) * value
        return torch.einsum('btkf,btkfo->btko', hidden_expert, output_weight)

class DeepSeekV4Gate(nn.Module):

    def __init__(self, model_dim=8, experts=256, top_k=6, token_to_expert=None):
        super().__init__()
        assert experts == 256
        assert top_k == 6
        self.experts = experts
        self.top_k = top_k
        self.weight = nn.Parameter(torch.randn(experts, model_dim) * 0.02)
        self.correction_bias = nn.Parameter(torch.zeros(experts), requires_grad=False)
        if token_to_expert is None:
            self.register_buffer('token_to_expert', None)
        else:
            assert token_to_expert.size(-1) == top_k
            self.register_buffer('token_to_expert', token_to_expert.long())

    @property
    def uses_hash_routing(self):
        return self.token_to_expert is not None

    def forward(self, hidden, token_ids=None):
        logits = F.linear(hidden.float(), self.weight.float())
        affinity = torch.sqrt(F.softplus(logits))
        if self.uses_hash_routing:
            if token_ids is None:
                raise ValueError('hash-routed layer requires token_ids')
            expert_ids = self.token_to_expert[token_ids]
        else:
            expert_ids = (affinity + self.correction_bias).topk(self.top_k, dim=-1).indices
        selected = affinity.gather(-1, expert_ids)
        weight = selected / selected.sum(dim=-1, keepdim=True).clamp_min(1e-08)
        return (weight.to(hidden.dtype), expert_ids)

class DeepSeekV4MoE(nn.Module):

    def __init__(self, model_dim=8, intermediate_dim=8, token_to_expert=None):
        super().__init__()
        self.experts = 256
        self.top_k = 6
        self.shared_expert_count = 1
        self.gate = DeepSeekV4Gate(model_dim=model_dim, token_to_expert=token_to_expert)
        self.routed = BatchedSwiGLUExperts(256, model_dim, intermediate_dim, model_dim)
        self.shared = BatchedSwiGLUExperts(1, model_dim, intermediate_dim, model_dim)

    def forward(self, hidden, token_ids=None):
        weight, expert_ids = self.gate(hidden, token_ids)
        routed = self.routed.selected_forward(hidden, expert_ids)
        routed = (routed * weight[..., None]).sum(dim=2)
        shared_ids = torch.zeros(
            hidden.size(0), hidden.size(1), 1,
            dtype=torch.long, device=hidden.device,
        )
        shared = self.shared.selected_forward(hidden, shared_ids).squeeze(2)
        return routed + shared


## 2. V4 decoder block primitive used by both main and MTP paths

The released MTP class is a decoder `Block` subclass. Therefore the reduced implementation below
keeps one decoder-block class and lets MTP invoke that class rather than defining a cheaper
MTP-only attention block.


In [ ]:
def simple_sinkhorn(logits, iterations=20, eps=1e-06):
    matrix = logits.softmax(dim=-1) + eps
    matrix = matrix / (matrix.sum(dim=-2, keepdim=True) + eps)
    for _ in range(iterations - 1):
        matrix = matrix / (matrix.sum(dim=-1, keepdim=True) + eps)
        matrix = matrix / (matrix.sum(dim=-2, keepdim=True) + eps)
    return matrix

class HyperConnection(nn.Module):

    def __init__(self, model_dim=8, streams=4):
        super().__init__()
        self.streams = streams
        output = (2 + streams) * streams
        self.mapping = nn.Linear(streams * model_dim, output, bias=True)
        self.scale = nn.Parameter(torch.full((3,), 0.01))

    def forward(self, hidden_streams):
        flat = hidden_streams.flatten(2)
        mapped = self.mapping(F.rms_norm(flat, (flat.size(-1),)))
        streams = self.streams
        pre, post, comb = mapped.split([streams, streams, streams * streams], dim=-1)
        pre_scale, post_scale, comb_scale = self.scale
        pre = torch.sigmoid(pre * pre_scale)
        post = 2.0 * torch.sigmoid(post * post_scale)
        comb = comb.view(*comb.shape[:-1], streams, streams)
        comb = simple_sinkhorn(comb * comb_scale, iterations=20)
        collapsed = (pre.unsqueeze(-1) * hidden_streams).sum(dim=2)
        return (post, comb, collapsed)

def apply_hyper_connection(hidden_streams, branch, connection):
    post, comb, collapsed = connection(hidden_streams)
    branch_output = branch(collapsed)
    mixed = torch.einsum('btij,btjd->btid', comb, hidden_streams)
    written = post.unsqueeze(-1) * branch_output.unsqueeze(2)
    return mixed + written

def rope_frequency(position_ids, rope_dim, base=10000.0):
    pair_index = torch.arange(0, rope_dim, 2, device=position_ids.device, dtype=torch.float32)
    inverse_frequency = 1.0 / base ** (pair_index / rope_dim)
    return position_ids.float()[:, None] * inverse_frequency[None]

def rotate_partial_rope(x, position_ids, rope_dim, conjugate=False):
    content = x[..., :-rope_dim]
    rotary = x[..., -rope_dim:]
    angle = rope_frequency(position_ids, rope_dim)
    cosine = angle.cos()[None, None]
    sine = angle.sin()[None, None]
    if conjugate:
        sine = -sine
    even = rotary[..., 0::2]
    odd = rotary[..., 1::2]
    rotated_even = even * cosine - odd * sine
    rotated_odd = even * sine + odd * cosine
    rotated = torch.stack([rotated_even, rotated_odd], dim=-1).flatten(-2)
    return torch.cat([content, rotated], dim=-1)

class V4GroupedOutputProjection(nn.Module):

    def __init__(self, query_heads=64, head_dim=4, groups=8, low_rank=8, model_dim=8):
        super().__init__()
        total_width = query_heads * head_dim
        assert total_width % groups == 0
        self.groups = groups
        self.group_width = total_width // groups
        self.down = nn.ModuleList([
            nn.Linear(self.group_width, low_rank, bias=False)
            for _ in range(groups)
        ])
        self.up = nn.Linear(groups * low_rank, model_dim, bias=False)

    def forward(self, heads):
        flattened = heads.flatten(2)
        chunks = flattened.split(self.group_width, dim=-1)
        latent_chunks = [projection(chunk) for projection, chunk in zip(self.down, chunks)]
        return self.up(torch.cat(latent_chunks, dim=-1))

class V4SharedKVAttention(nn.Module):

    def __init__(self, model_dim=8, query_heads=64, head_dim=4, rope_dim=2, q_rank=4, window=128):
        super().__init__()
        assert query_heads == 64
        assert window == 128
        assert 0 < rope_dim < head_dim
        assert rope_dim % 2 == 0
        self.query_heads = query_heads
        self.head_dim = head_dim
        self.rope_dim = rope_dim
        self.window = window
        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_norm = nn.RMSNorm(q_rank)
        self.q_up = nn.Linear(q_rank, query_heads * head_dim, bias=False)
        self.shared_kv = nn.Linear(model_dim, head_dim, bias=False)
        self.shared_kv_norm = nn.RMSNorm(head_dim)
        self.sink = nn.Parameter(torch.zeros(query_heads))
        self.output_projection = V4GroupedOutputProjection(
            query_heads=query_heads, head_dim=head_dim, groups=8,
            low_rank=8, model_dim=model_dim,
        )

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        positions = torch.arange(length, device=hidden.device)
        q_latent = self.q_norm(self.q_down(hidden))
        query = self.q_up(q_latent)
        query = query.view(
            batch, length, self.query_heads, self.head_dim
        ).transpose(1, 2)
        query = F.rms_norm(query, (self.head_dim,))
        query = rotate_partial_rope(query, positions, self.rope_dim)
        shared_kv = self.shared_kv_norm(self.shared_kv(hidden))
        shared_kv = rotate_partial_rope(shared_kv[:, None], positions, self.rope_dim).squeeze(1)
        outputs = []
        for token_index in range(length):
            start = max(0, token_index - self.window + 1)
            key_value = shared_kv[:, start:token_index + 1]
            score = torch.einsum(
                'bhd,bkd->bhk', query[:, :, token_index], key_value
            ) / math.sqrt(self.head_dim)
            sink = self.sink[None, :, None].expand(batch, -1, 1)
            weight = torch.cat([score, sink], dim=-1).softmax(dim=-1)[..., :-1]
            output = torch.einsum('bhk,bkd->bhd', weight, key_value)
            outputs.append(output)
        heads = torch.stack(outputs, dim=2)
        heads = rotate_partial_rope(heads, positions, self.rope_dim, conjugate=True)
        heads = heads.transpose(1, 2).contiguous()
        return self.output_projection(heads)

class DeepSeekV4DecoderBlock(nn.Module):

    def __init__(self, model_dim=8, token_to_expert=None):
        super().__init__()
        self.attention_hc = HyperConnection(model_dim, streams=4)
        self.ffn_hc = HyperConnection(model_dim, streams=4)
        self.attention = V4SharedKVAttention(
            model_dim=model_dim, query_heads=64, head_dim=4,
            rope_dim=2, window=128,
        )
        self.moe = DeepSeekV4MoE(model_dim=model_dim, token_to_expert=token_to_expert)

    def forward(self, hidden_streams, token_ids=None):
        hidden_streams = apply_hyper_connection(hidden_streams, self.attention, self.attention_hc)

        def moe_branch(hidden):
            return self.moe(hidden, token_ids=token_ids)
        hidden_streams = apply_hyper_connection(hidden_streams, moe_branch, self.ffn_hc)
        return hidden_streams


## 3. MTP: normalized fusion + one real V4 decoder block + shared embedding/head


In [ ]:
class DeepSeekV4MTPDepth(nn.Module):

    def __init__(self, shared_embedding, shared_lm_head, model_dim=8):
        super().__init__()
        self.enorm = nn.RMSNorm(model_dim)
        self.hnorm = nn.RMSNorm(model_dim)
        self.eh_projection = nn.Linear(2 * model_dim, model_dim, bias=False)
        self.block = DeepSeekV4DecoderBlock(model_dim=model_dim)
        self.input_expansion = nn.Linear(model_dim, 4 * model_dim, bias=False)
        self.head_norm = nn.RMSNorm(model_dim)
        self.shared_embedding = shared_embedding
        self.shared_lm_head = shared_lm_head

    def collapse_streams(self, streams):
        return streams.mean(dim=2)

    def forward(self, main_hidden, shifted_token_ids):
        shifted_embedding = self.shared_embedding(shifted_token_ids)
        fused_input = torch.cat(
            [self.hnorm(main_hidden), self.enorm(shifted_embedding)], dim=-1
        )
        fused = self.eh_projection(fused_input)
        batch, length, dim = fused.shape
        streams = self.input_expansion(fused).view(batch, length, 4, dim)
        streams = self.block(streams)
        hidden = self.collapse_streams(streams)
        hidden = self.head_norm(hidden)
        return self.shared_lm_head(hidden)
vocab = 32
model_dim = 8
shared_embedding = nn.Embedding(vocab, model_dim).to(device)
shared_lm_head = nn.Linear(model_dim, vocab, bias=False).to(device)
mtp = DeepSeekV4MTPDepth(shared_embedding, shared_lm_head, model_dim=model_dim).to(device)
assert isinstance(mtp.block, DeepSeekV4DecoderBlock)
assert mtp.block.attention.query_heads == 64
assert mtp.block.attention.window == 128
assert mtp.block.moe.experts == 256
assert mtp.block.moe.top_k == 6
assert mtp.block.attention_hc.streams == 4
assert mtp.block.ffn_hc.streams == 4
main_hidden = torch.randn(1, 3, model_dim)
shifted_ids = torch.randint(0, vocab, (1, 3))
logits = mtp(main_hidden, shifted_ids)
logits.square().mean().backward()
assert logits.shape == (1, 3, vocab)


## Audit result

The former MTP-only simplified attention class is gone. MTP now runs the same reduced V4 decoder
block type—with 4-stream mHC, 64-head shared-KV local attention, and 256/top-6/+1 MoE—after the
released normalized hidden/token-embedding fusion.
